# 🦵 RSNA Knee Abnormality Detection — EDA

> **Competition:** [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)  
> **Task:** Multi-label binary classification of 12 knee abnormalities from multimodal MRI  
> **Metric:** Macro-averaged AUC-ROC across 12 targets  
> **Data:** 570 GB of DICOM MRI series + radiology reports (multilingual)  

## What's covered here
1. Dataset structure — studies, series, files
2. Label distribution — class imbalance, co-occurrence
3. Series metadata — planes, fluid sensitivity, fat suppression
4. DICOM properties — intensity, shape, pixel spacing
5. Radiology reports — language distribution, length, keyword patterns
6. Sample MRI slices — visual inspection across planes


In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = sns.color_palette('mako', 12)

BASE = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
print('Files available:', list(BASE.glob('*.csv')))

## 1. Dataset Overview

In [ ]:
train = pd.read_csv(BASE / 'train.csv')
train_series = pd.read_csv(BASE / 'train_series.csv')
test  = pd.read_csv(BASE / 'test.csv')
test_series  = pd.read_csv(BASE / 'test_series.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')

TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
           'Medial OA','Lateral OA','PF OA','Effusion',
           'Synovitis',"Baker's",'Contusion','Fracture']

# Print study and series lengths
print(f'Training studies:         {len(train):,}')
print(f'Training series:          {len(train_series):,}')
print(f'Test studies:             {len(test):,}')
print(f'Test series:              {len(test_series):,}')
print(f'Series per study (train): {len(train_series)/len(train):.1f} avg')
# Display columns
print('\nTrain columns:', train.columns.tolist())
# Display the first 3 studies that actually have labels
train[train[TARGETS].notna().all(axis=1)].head(3)

### 1.5 Missing Data Analysis

In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': train.isnull().sum(),
    'Missing Percentage (%)': (train.isnull().mean() * 100).round(2)
})
print("Missing data in train.csv:")
display(missing_summary)
# Calculate exact label completeness
labeled_count = train[TARGETS].notna().all(axis=1).sum()
print(f"\nStudies with labels: {labeled_count} ({labeled_count/len(train)*100:.2f}%)")
print(f"Studies without labels (Report only): {len(train) - labeled_count} ({(len(train)-labeled_count)/len(train)*100:.2f}%)")

## 2. Label Distribution & Class Imbalance

In [ ]:
# How many studies have labels vs only reports?
labeled = train[TARGETS].notna().all(axis=1)
print(f'Studies with full labels: {labeled.sum():,} / {len(train):,} ({labeled.mean()*100:.1f}%)')
print(f'Studies with report only: {(~labeled).sum():,}')

df_lab = train[labeled].copy()
pos_rates = df_lab[TARGETS].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Positive rate per class
ax = axes[0]
bars = ax.barh(pos_rates.index, pos_rates.values * 100,
               color=sns.color_palette('mako_r', len(TARGETS)))
ax.set_xlabel('Positive Rate (%)')
ax.set_title('Positive Rate per Abnormality', fontweight='bold')
for bar, val in zip(bars, pos_rates.values):
    ax.text(val * 100 + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=9)

# Labels per study
ax = axes[1]
labels_per_study = df_lab[TARGETS].sum(axis=1)
labels_per_study.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Number of Abnormalities per Study')
ax.set_ylabel('Count')
ax.set_title('Abnormality Count Distribution', fontweight='bold')
ax.tick_params(axis='x', rotation=0)


plt.show()
print(f'\nMean abnormalities per labeled study: {labels_per_study.mean():.2f}')
print(f'Studies with 0 abnormalities: {(labels_per_study==0).sum():,}')

In [ ]:
# Co-occurrence heatmap
fig, ax = plt.subplots(figsize=(11, 9))
co = df_lab[TARGETS].T.dot(df_lab[TARGETS])
# Normalize to conditional probability P(col | row)
co_norm = co.div(co.values.diagonal(), axis=0)
mask = np.eye(len(TARGETS), dtype=bool)
sns.heatmap(co_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'P(col | row)'})
ax.set_title('Abnormality Co-occurrence (P(column | row))', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 3. Patient Metadata

## 4. MRI Series Metadata

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Anatomical plane
plane_counts = train_series['Anatomical_Plane'].value_counts()
axes[0].bar(plane_counts.index, plane_counts.values,
            color=['#4e79a7','#f28e2b','#59a14f'])
axes[0].set_title('Series by Anatomical Plane', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(plane_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Fluid sensitive
fs = train_series['Fluid_Sensitive'].value_counts()
axes[1].bar(['Non-Fluid\nSensitive','Fluid\nSensitive'], [fs.get(0,0), fs.get(1,0)],
            color=['#86b5d9','#2a5f8f'])
axes[1].set_title('Fluid Sensitivity Flag', fontweight='bold')

# Fat suppression
fat = train_series['Fat_Suppression'].value_counts()
axes[2].bar(['No Fat\nSuppression','Fat\nSuppression'], [fat.get(0,0), fat.get(1,0)],
            color=['#f0a868','#c75b2a'])
axes[2].set_title('Fat Suppression Flag', fontweight='bold')

plt.tight_layout()
plt.show()

# Series per study
sps = train_series.groupby('StudyInstanceUID').size()
print(f'Series per study — min:{sps.min()} | median:{sps.median():.0f} | max:{sps.max()}')

# Combination breakdown
combos = train_series.groupby(['Anatomical_Plane','Fluid_Sensitive','Fat_Suppression']).size().reset_index(name='Count').sort_values('Count', ascending=False)
print('\nTop 10 series type combinations:')
display(combos.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sps.values, bins=30, color='steelblue', edgecolor='white')
ax.axvline(sps.median(), color='red', linestyle='--', label=f'Median = {sps.median():.0f}')
ax.set_xlabel('Series per Study')
ax.set_ylabel('# Studies')
ax.set_title('Distribution of Series per Study', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Radiology Reports — Text Analysis

In [ ]:
# Report length
train['report_len'] = train['Report'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train['report_len'], bins=50, color='mediumpurple', edgecolor='white')
axes[0].axvline(train['report_len'].median(), color='red', linestyle='--',
                label=f"Median = {train['report_len'].median():.0f} words")
axes[0].set_xlabel('Report Length (words)')
axes[0].set_ylabel('Studies')
axes[0].set_title('Radiology Report Length Distribution', fontweight='bold')
axes[0].legend()

# Language detection (simple heuristic)
def detect_lang(text):
    if not isinstance(text, str) or len(text.strip()) < 10:
        return 'empty'
    # Simple n-gram heuristics
    t = text.lower()
    if any(w in t for w in ['the ','and ','of the','is ','are ']):
        return 'English'
    elif any(w in t for w in ['la ','le ','et ','du ','des ']):
        return 'French'
    elif any(w in t for w in ['der ','die ','das ','und ','mit ']):
        return 'German'
    elif any(w in t for w in [' el ',' la ',' en ',' de ',' que ']):
        return 'Spanish'
    elif any(w in t for w in [' il ',' di ',' che ',' un ',' una ']):
        return 'Italian'
    return 'Other'

train['lang'] = train['Report'].apply(detect_lang)
lang_counts = train['lang'].value_counts()
axes[1].bar(lang_counts.index, lang_counts.values,
            color=sns.color_palette('Set2', len(lang_counts)))
axes[1].set_title('Estimated Report Language', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(lang_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Top keywords in reports that correlate with each label
KEYWORDS = {
    'ACL': ['acl','anterior cruciate','ligament tear'],
    'MCL': ['mcl','medial collateral'],
    'Medial Meniscus': ['medial meniscus','medial horn'],
    'Lateral Meniscus': ['lateral meniscus','lateral horn'],
    'Effusion': ['effusion','fluid','joint fluid'],
    'Fracture': ['fracture','stress fracture','break'],
    'Contusion': ['contusion','bone bruise','edema'],
    "Baker's": ['baker','popliteal cyst','cyst'],
}

if labeled.sum() > 0:
    rows = []
    for label, kws in KEYWORDS.items():
        if label not in df_lab.columns:
            continue
        pos = df_lab[df_lab[label] == 1]['Report'].fillna('').str.lower()
        neg = df_lab[df_lab[label] == 0]['Report'].fillna('').str.lower()
        for kw in kws:
            rows.append({
                'Label': label,
                'Keyword': kw,
                'Rate in Positives': pos.str.contains(kw).mean(),
                'Rate in Negatives': neg.str.contains(kw).mean()
            })
    kw_df = pd.DataFrame(rows)
    kw_df['Lift'] = kw_df['Rate in Positives'] / (kw_df['Rate in Negatives'] + 1e-6)
    print('Keyword signal in radiology reports (sample):')
    display(kw_df.sort_values('Lift', ascending=False).head(15))

## 6. DICOM Inspection — Sample Slices

In [ ]:
try:
    import pydicom
    PYDICOM = True
except ImportError:
    print('pydicom not available — skipping DICOM visualization')
    PYDICOM = False

if PYDICOM:
    def load_dcm(path):
        ds = pydicom.dcmread(str(path))
        img = ds.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)
        return img, ds

    # Load example studies from test set (no labels needed)
    test_dcms = sorted((BASE / 'test_series').rglob('*.dcm'))
    print(f'Test DICOM files available: {len(test_dcms)}')

In [ ]:
if PYDICOM and len(test_dcms) > 0:
    # Group by series — pick 3 series from first study
    series_paths = {}
    for p in test_dcms:
        sid = p.parent.name
        series_paths.setdefault(sid, []).append(p)

    fig = plt.figure(figsize=(18, 5 * min(3, len(series_paths))))
    for row_idx, (sid, paths) in enumerate(list(series_paths.items())[:3]):
        paths = sorted(paths)
        n_slices = len(paths)
        # Show 5 evenly spaced slices
        indices = np.linspace(0, n_slices - 1, 5, dtype=int)
        for col_idx, i in enumerate(indices):
            ax = fig.add_subplot(min(3, len(series_paths)), 5, row_idx * 5 + col_idx + 1)
            try:
                img, ds = load_dcm(paths[i])
                ax.imshow(img, cmap='gray', aspect='auto')
                if col_idx == 0:
                    ax.set_ylabel(f'Series\n{sid[:20]}...', fontsize=7)
                ax.set_title(f'Slice {i+1}/{n_slices}', fontsize=7)
                ax.axis('off')
            except Exception as e:
                ax.text(0.5, 0.5, str(e)[:40], ha='center', va='center', transform=ax.transAxes, fontsize=6)
                ax.axis('off')

    plt.suptitle('Sample MRI Slices (Test Set — No Labels)', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    # DICOM metadata summary
    dcm_meta = []
    for p in test_dcms[:30]:
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            row = {
                'Rows': getattr(ds, 'Rows', None),
                'Columns': getattr(ds, 'Columns', None),
                'PixelSpacing': str(getattr(ds, 'PixelSpacing', None)),
                'SliceThickness': getattr(ds, 'SliceThickness', None),
                'Modality': getattr(ds, 'Modality', None),
                'TransferSyntax': str(getattr(ds, 'file_meta', {}).get('TransferSyntaxUID', None))
            }
            dcm_meta.append(row)
        except:
            pass

    meta_df = pd.DataFrame(dcm_meta)
    print('\nDICOM Metadata Summary (first 30 slices):')
    display(meta_df.describe())
    display(meta_df.head(5))

## 7. Key Takeaways for Modelling

### Summary

| Finding | Implication |
|---------|-------------|
| Only a fraction of training studies have hard labels | Use **report-based pseudo-labelling** (NLP on the radiology text) for the remainder |
| Multilingual reports | Need multilingual NLP (multilingual BERT, mDeBERTa, or GPT-based extraction) |
| 12 targets with varying prevalence | Use **per-label weighted BCE loss** or **focal loss** to handle imbalance |
| 3 imaging planes per study (Sagittal/Coronal/Axial) + fluid/fat flags | Plane-aware model — train separate CNN heads per plane or use attention-based fusion |
| Series are 3D volumes (20–45 slices, up to hundreds) | Use **2.5D slicing** (stack N adjacent slices), **3D convolutions**, or **slice sampling** |
| Metric = macro-averaged AUC | All 12 labels equally matter; use **threshold-free** losses (BCE with logits) |
| Multimodal: images + text | **Fusion models** (image encoder + text encoder) likely outperform image-only |

### Suggested Baseline Architecture
```
per-series: EfficientNetV2 / ConvNeXt / ViT (2.5D)
  → pool over slices → per-series embedding
per-study: attention pool over all series embeddings
  → study embedding
text head: BioClinicalBERT / mDeBERTa on report
  → report embedding
fusion: concat image + text embeddings → 12-head sigmoid output
loss: weighted BCE per label
```
